# Treinamento de modelos de falência usando o Banco SQLite

Este notebook integra o banco de dados ao treinamento de Machine Learning. Ele usa **somente os cinco CSVs locais** (`dados/1year.csv` até `dados/5year.csv`), importa os dados para SQLite, verifica a qualidade, compara Random Forest e XGBoost, mostra as métricas e grava os modelos.

**Execute as células em ordem e abra o notebook a partir da pasta raiz `entrega_falencia/`.** O treinamento final recria os arquivos de modelo e resultados da pasta do projeto.

## 1. Instalar bibliotecas

Se já executou `pip install -r requirements.txt` no terminal, pode pular esta célula.

In [ ]:
# Se necessário, descomente e execute:
# %pip install -r requirements.txt

## 2. Definir os caminhos e importar os CSVs para o banco

As tabelas por horizonte são preservadas separadamente. Os valores ausentes permanecem como `NULL` no banco; a imputação ocorre dentro do pipeline de treinamento.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from sqlalchemy import create_engine, inspect, text
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "criar_banco.py").exists():
    raise FileNotFoundError(
        "Abra o notebook a partir da pasta raiz entrega_falencia, onde está criar_banco.py."
    )
PASTA_DADOS = ROOT / "dados"
BANCO = ROOT / "falencia_empresas.db"

from criar_banco import criar_banco
resumo_importacao = criar_banco(PASTA_DADOS, BANCO)
resumo_importacao

## 3. Conferir as tabelas e as dimensões dos dados

In [ ]:
engine = create_engine(f"sqlite:///{BANCO.as_posix()}")
inspector = inspect(engine)
print("Tabelas SQL:", inspector.get_table_names())

linhas = []
for ano in range(1, 6):
    tabela = f"falencia_{ano}_ano"
    with engine.connect() as con:
        total = con.execute(text(f'SELECT COUNT(*) FROM "{tabela}"')).scalar_one()
    linhas.append({"horizonte_anos": ano, "tabela": tabela, "registros": total,
                   "colunas": len(inspector.get_columns(tabela))})

display(pd.DataFrame(linhas))

## 4. Qualidade dos dados e distribuição da variável-alvo

A classe `0` representa empresa não falida e a classe `1` representa empresa falida. A grande diferença de quantidade entre as classes importa na escolha das métricas: acurácia isolada pode parecer alta mesmo quando o modelo quase nunca identifica falências.

In [ ]:
qualidade = []
distribuicao = []
for ano in range(1, 6):
    tabela = f"falencia_{ano}_ano"
    with engine.connect() as con:
        df = pd.read_sql_query(f'SELECT * FROM "{tabela}"', con)
    cols_atributos = [f"Attr{i}" for i in range(1, 65)]
    qualidade.append({
        "horizonte_anos": ano,
        "registros": len(df),
        "colunas": len(df.columns),
        "valores_ausentes_atributos": int(df[cols_atributos].isna().sum().sum()),
        "percentual_falidas": round(float((df["class"] == 1).mean() * 100), 3),
    })
    vc = df["class"].value_counts(dropna=False)
    for classe in [0, 1]:
        distribuicao.append({"horizonte_anos": ano, "classe": classe,
                             "descricao": "Não falida" if classe == 0 else "Falida",
                             "quantidade": int(vc.get(classe, 0))})

display(pd.DataFrame(qualidade))
display(pd.DataFrame(distribuicao))

## 5. Observações sobre o notebook de análise original

O arquivo `dados/AnlisedeDados_original.ipynb` foi preservado como referência. Ele concatena os cinco CSVs, explora correlações e distribuições e preenche valores ausentes com medianas para gráficos exploratórios. Essa transformação é útil para EDA, mas não deve ser reutilizada diretamente antes da divisão treino/teste de um experimento preditivo: as medianas precisam ser aprendidas somente no conjunto de treino.

O novo script corrige esse ponto usando `SimpleImputer(strategy="median")` dentro de cada `Pipeline`, realiza a seleção dos 20 atributos apenas no conjunto de treino e usa o conjunto de validação para escolher o limiar de classificação.

## 6. Treinar, comparar e salvar os modelos

A execução é reproduzível (`random_state=42`). A divisão é 60% treino, 20% validação e 20% teste. O modelo final é depois retreinado com todos os dados disponíveis para uso no sistema; as métricas apresentadas continuam sendo as obtidas no conjunto de teste reservado.

In [ ]:
from treinar_modelos import main as treinar_modelos

resultados = treinar_modelos(
    pasta_csv=None,
    pasta_saida=ROOT,
    caminho_banco=BANCO,
)
print("Modelo selecionado:", resultados["melhor_modelo_selecionado"])
print("Limiar:", resultados["limiar_final"])
print("Atributos selecionados:", resultados["atributos_selecionados"])

## 7. Tabela comparativa no teste

In [ ]:
comparacao = pd.read_csv(ROOT / "comparacao_modelos.csv")
display(comparacao.round(4))

## 8. Matrizes de confusão

In [ ]:
for nome, metricas in resultados["resultados_teste"].items():
    print(f"\n{nome}")
    print("Matriz de confusão (linhas=real; colunas=prevista, ordem [0, 1]):")
    display(pd.DataFrame(
        metricas["matriz_confusao"],
        index=["Real: não falida", "Real: falida"],
        columns=["Prevista: não falida", "Prevista: falida"],
    ))
    print({k: round(metricas[k], 4) for k in [
        "acuracia", "precisao_falencia", "recall_falencia", "f1_falencia",
        "roc_auc", "average_precision", "limiar"
    ]})

## 9. Arquivos gerados e cuidados de interpretação

Ao terminar, os artefatos são gravados na raiz do projeto: `modelo_falencia.joblib`, `modelo_random_forest.joblib`, `modelo_xgboost.joblib`, `comparacao_modelos.csv`, `resultados_treinamento.json` e `atributos_selecionados.csv`.

O recall de falência mede a proporção de empresas falidas que o modelo conseguiu sinalizar; a precisão mede quantas das empresas sinalizadas realmente eram falidas no conjunto de teste. Como a classe positiva é rara e os falsos positivos ainda podem ser numerosos, o resultado serve como apoio à priorização de análises, não como decisão financeira automatizada.